<a href="https://colab.research.google.com/github/disha-002/DeepLearning/blob/main/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

np.random.seed(42)

X=np.random.uniform(-2,2,(400,3))

y = (np.sin(X[:,0])+ 0.5*(X[:,1]**2)- 0.8*X[:,2])

y = y.reshape(-1,1)

Total Parameters

Model A

layer 1: (3+1)*4 = 16

layer 2: (4+1)*1 = 5

**total = 31**

Model B

layer 1: (3+1)*6 = 24

layer 2: (6+1)*6 = 42

layer 3: (6+1)*1 = 7

**total = 73**

Model C

layer 1: (3+1)*8 = 32

layer 2: (8+1)*8 = 72

layer 3: (8+1)*8 = 72

layer 4: (8+1)*8 = 72

layer 5: (8+1)*1 = 9

**total = 257**

Model D

layer 1: (3+1)*8 = 32

layer 2: (8+1)*8 = 72

layer 3: (8+1)*8 = 72

layer 4: (8+1)*8 = 72

layer 5: (8+1)*8 = 72

layer 6: (8+1)*8 = 72

layer 7: (8+1)*8 = 72

layer 8: (8+1)*8 = 72

layer 9: (8+1)*1 = 9

**total = 545**


In [2]:
def Relu(z):
  return np.maximum(0,z)

def Sigmoid(z):
  return 1/(1+np.exp(-z))

def Tanh(z):
  return (np.exp(z)-np.exp(-z))/(np.exp(z)+np.exp(-z))

def Leaky_Relu(z,a=0.01):
  return np.maximum(a*z,z)

def softplus(z):
  return np.log(1+np.exp(z))



In [3]:
def Relu_slope(z):
  return (z>0).astype(float)

def Sigmoid_slope(z):
  s = Sigmoid(z)
  return s*(1-s)

def Tanh_slope(z):
  t = Tanh(z)
  return 1-t**2

def Leaky_Relu_slope(z,a=0.01):
  return np.where(z>0,1,a)

def softplus_slope(z):
  return Sigmoid(z)

In [4]:
def initialization(layers):
  parameters = {}
  for i in range(1,len(layers)):
    parameters["W"+str(i)]= np.random.uniform(-0.5,0.5,(layers[i],layers[i-1]))
    parameters["b"+str(i)]= np.zeros((layers[i],1))

  return parameters

In [5]:
def forward_pass(X,parameters,activation):
  cache = {}
  A = X.T
  cache["A0"] = A

  L = len(parameters)//2

  for l in range(1, L):
      Z = parameters["W"+str(l)] @ A + parameters["b"+str(l)]
      A = activation(Z)
      cache["Z"+str(l)] = Z
      cache["A"+str(l)] = A


  ZL = parameters["W"+str(L)] @ A + parameters["b"+str(L)]
  cache["Z"+str(L)] = ZL
  cache["A"+str(L)] = ZL

  return ZL.T, cache

In [6]:
def mse(y, y_hat):
    return np.mean((y - y_hat)**2)

In [7]:
def backward_pass(X, y, params, cache, activation_deriv):
    grads = {}
    N = X.shape[0]
    L = len(params)//2

    y_hat = cache["A"+str(L)].T
    dL_dy = 2*(y_hat - y)/N # loss wrt y_hat
    dL_dy = dL_dy.T


    grads["dW"+str(L)] = dL_dy @ cache["A"+str(L-1)].T
    grads["db"+str(L)] = np.sum(dL_dy, axis=1, keepdims=True)

    dL_dh = params["W"+str(L)].T @ dL_dy


    for l in reversed(range(1, L)):
        dL_dz = dL_dh * activation_deriv(cache["Z"+str(l)])
        grads["dW"+str(l)] = dL_dz @ cache["A"+str(l-1)].T
        grads["db"+str(l)] = np.sum(dL_dz, axis=1, keepdims=True)
        dA_prev = params["W"+str(l)].T @ dL_dz

    return grads

In [8]:
def update(params, grads, lr=0.01):
    L = len(params)//2
    for l in range(1, L+1):
        params["W"+str(l)] -= lr * grads["dW"+str(l)]
        params["b"+str(l)] -= lr * grads["db"+str(l)]
    return params

In [9]:
def grad_norm(matrix):
    return np.sqrt(np.sum(matrix**2))

In [10]:
import pandas as pd

columns = [
    "Model",
    "Activation",
    "Final Loss",
    "Grad Norm L1",
    "Grad Norm Last",
    "Observation"
]

results = pd.DataFrame(columns=columns)

results

,Model,Activation,Final Loss,Grad Norm L1,Grad Norm Last,Observation


In [11]:
def train(layers, activation, activation_deriv):

    params = initialization(layers)
    epochs = 1000

    for epoch in range(epochs):
        y_hat, cache = forward_pass(X, params, activation)
        loss = mse(y, y_hat)

        grads = backward_pass(X, y, params, cache, activation_deriv)
        params = update(params, grads, 0.01)

        if epoch == 200:
          print("Loss after 200 epochs:", loss)

    # AFTER training finishes
    L = len(params)//2

    g1 = grad_norm(grads["dW1"])
    g_last = grad_norm(grads["dW"+str(L-1)])

    print("Final Loss:", loss)
    print("Gradient Norm Layer 1:", g1)
    print("Gradient Norm Last Hidden:", g_last)

    return loss, g1, g_last

In [12]:
loss, g1, g_last=train([3,4,1], Relu, Relu_slope)


Loss after 200 epochs: 0.49268594052687364
Final Loss: 0.11154512827725767
Gradient Norm Layer 1: 0.04521736074874136
Gradient Norm Last Hidden: 0.04521736074874136


In [13]:
results.loc[len(results)] = ["Model A","ReLU",loss,g1,g_last,"Stable"]

In [14]:
loss, g1, g_last = train([3,6,6,1], Relu, Relu_slope)


Loss after 200 epochs: 0.5282822770028545
Final Loss: 0.0792172217780415
Gradient Norm Layer 1: 0.06518592894137455
Gradient Norm Last Hidden: 0.031438301269799115


In [15]:
results.loc[len(results)] = ["Model B","ReLU",loss,g1,g_last,"Stable"]

In [16]:
loss, g1, g_last = train([3,8,8,8,8,1],Relu, Relu_slope)



Loss after 200 epochs: 0.5377904922787162
Final Loss: 0.061035982845135645
Gradient Norm Layer 1: 0.0358863126137341
Gradient Norm Last Hidden: 0.03335857241874416


In [17]:
results.loc[len(results)] = ["Model C","ReLU",loss,g1,g_last,"Slow"]

In [18]:
loss, g1, g_last = train([3,8,8,8,8,8,8,8,8,1], Relu, Relu_slope)


Loss after 200 epochs: 0.5190580660917862
Final Loss: 0.1580953366380895
Gradient Norm Layer 1: 0.1472350666180168
Gradient Norm Last Hidden: 0.04282632554598956


In [19]:
results.loc[len(results)] = ["Model D","ReLU",loss,g1,g_last,"slow"]

In [20]:
loss, g1, g_last = train([3,8,8,8,8,8,8,8,8,1], Sigmoid, Sigmoid_slope)

Loss after 200 epochs: 1.7438496981975375
Final Loss: 1.7438502187103797
Gradient Norm Layer 1: 0.1879773071117501
Gradient Norm Last Hidden: 4.321329887470623e-06


In [21]:
results.loc[len(results)] = ["Model D","Sigmoid",loss,g1,g_last,"unstable"]

In [22]:
results

,Model,Activation,Final Loss,Grad Norm L1,Grad Norm Last,Observation
0,Model A,ReLU,0.111545,0.045217,0.045217,Stable
1,Model B,ReLU,0.079217,0.065186,0.031438,Stable
2,Model C,ReLU,0.061036,0.035886,0.033359,Slow
3,Model D,ReLU,0.158095,0.147235,0.042826,slow
4,Model D,Sigmoid,1.743850,0.187977,0.000004,unstable


**Reflections**
1. No, deeper did not always reduce loss faster.

2. the gradient stayed the exact in model A , slights similar in model B and model C
and there was huge drop in model D both using Relu and Sigmoid activation function.

3. No, it wasn't stable for sigmoid.

4. Relu behaved more stable in deeper network.

5. yes , some models improved very slowly even when the learning rate was the same.